In [1]:
import sys
import pandas as pd
from pathlib import Path

ROOT = Path(r"d:\ML Project - E-commerce product search with semantic ranking")
DATA = ROOT / "data"
sys.path.append(str(ROOT))

from src.evaluate import evaluate_run

qrels = pd.read_parquet(DATA / "qrels.parquet")
bm25_run = pd.read_parquet(DATA / "run_bm25.parquet")

print("Judgments:", len(qrels))
print("BM25 run rows:", len(bm25_run))
bm25_run.head()

Judgments: 10125
BM25 run rows: 50000


,query_id,product_id,rank,score
0,53255,B003D122R2,1,25.115397
1,53255,B081ZTKK1W,2,16.811029
2,53255,B07V92QMC5,3,13.107329
3,53255,B08B8QXGT8,4,13.028410
4,53255,B088WNC2K9,5,12.327079


In [2]:
bm25_scores = evaluate_run(bm25_run, qrels, k=10)
bm25_scores

{'Recall@10': 0.3362,
 'MRR@10': 0.7332,
 'NDCG@10': 0.4952,
 'queries_evaluated': 500}

In [4]:
for k in [5, 10, 20]:
    print(k, evaluate_run(bm25_run, qrels, k=k))

5 {'Recall@5': 0.2044, 'MRR@5': 0.7268, 'NDCG@5': 0.5016, 'queries_evaluated': 500}
10 {'Recall@10': 0.3362, 'MRR@10': 0.7332, 'NDCG@10': 0.4952, 'queries_evaluated': 500}
20 {'Recall@20': 0.4853, 'MRR@20': 0.7367, 'NDCG@20': 0.5129, 'queries_evaluated': 500}


In [5]:
test_qrels = pd.DataFrame({
    "query_id": ["q1", "q1", "q1"],
    "product_id": ["a", "b", "c"],
    "gain": [3, 2, 0],
})

# Perfect ranking: best product first
perfect = pd.DataFrame({"query_id": ["q1"]*3, "product_id": ["a", "b", "c"], "rank": [1, 2, 3]})

# Bad ranking: irrelevant first
bad = pd.DataFrame({"query_id": ["q1"]*3, "product_id": ["c", "b", "a"], "rank": [1, 2, 3]})

print("Perfect:", evaluate_run(perfect, test_qrels, k=3))
print("Bad:    ", evaluate_run(bad, test_qrels, k=3))

Perfect: {'Recall@3': 1.0, 'MRR@3': 1.0, 'NDCG@3': 1.0, 'queries_evaluated': 1}
Bad:     {'Recall@3': 1.0, 'MRR@3': 0.5, 'NDCG@3': 0.6064, 'queries_evaluated': 1}


In [6]:
results = pd.DataFrame([{"Method": "BM25", **bm25_scores}])
results.to_csv(DATA / "results.csv", index=False)
results

,Method,Recall@10,MRR@10,NDCG@10,queries_evaluated
0,BM25,0.3362,0.7332,0.4952,500


In [7]:
from src.evaluate import build_qrels_dict, build_run_dict

queries = pd.read_parquet(DATA / "queries.parquet")
corpus = pd.read_parquet(DATA / "corpus.parquet")

qrels_d = build_qrels_dict(qrels)
run_d = build_run_dict(bm25_run)

# Queries where the top 10 contain NO relevant product
failed = []
for qid, judged in qrels_d.items():
    relevant = {p for p, g in judged.items() if g >= 2}
    if relevant and not (set(run_d.get(qid, [])[:10]) & relevant):
        failed.append(qid)

print(f"BM25 completely failed on {len(failed)} of {len(qrels_d)} queries")
queries[queries["query_id"].isin(failed)].head(15)

BM25 completely failed on 58 of 500 queries


,query_id,query
3,56046,jms580
57,16698,black out window curtain
59,42459,free fantasy books for kindle
64,74548,novacide
68,5879,a plus
75,25088,cheap jean jumper for woman
77,27311,cocodrilo de juguete
82,36649,ecografo
89,46210,go pro camera
104,80371,pinney adult xl
